# **Estructura estándar recomendada para `stage_XX_*.py`**


## **0. Docstring del módulo (contrato del stage)**


Debe incluir:
- Nombre del stage
- Propósito (1-2 líneas)
- Inputs / Outputs / Reports / Params
- Notas (supuestos y no-objetivos)

## **1. Imports (orden fijo)**

1. stdlib
2. third-party
3. local imports
4. imports opcionales dentro de funciones (MLflow, etc.)

## **2. Logging (uniforme)**

- `LOG_LEVEL` por env
- `log = logging.getLogger("stage_XX")`

In [ ]:
import logging
import os

logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s"
)
log = logging.getLogger("stage_01")

## **3. Configuración DVC-friendly (siempre presente)**

Siempre definir rutas como constantes (sin “inventarlas” en argparse):
- `IN_*` (deps)
- `OUT_*` (outs)
- `REPORT_*` (reports)

Ejemplo:

In [ ]:
from pathlib import Path
import os

# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
IN_RAW_PARQUET  = Path(os.environ.get("IN_RAW_PARQUET", "data/raw/mnq_raw.parquet"))
OUT_PARQUET     = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday.parquet"))
REPORT_SUMMARY  = Path(os.environ.get("REPORT_SUMMARY", "reports/stage_01_dataset_prep_summary.json"))


## **4. Configuración funcional (parámetros del stage)**

Bloque “Params” con defaults reproducibles (env-first):

In [ ]:
MARKET = os.environ.get("MARKET", "NASDAQ")
TZ_FROM = os.environ.get("TZ_FROM", "UTC")
TZ_TO = os.environ.get("TZ_TO", "America/New_York")
TRADING_START = os.environ.get("TRADING_START", "06:30:00")
TRADING_END = os.environ.get("TRADING_END", "16:00:00")
GAP_MINUTES = int(os.environ.get("GAP_MINUTES", "1"))
ENABLE_MLFLOW = os.environ.get("ENABLE_MLFLOW", "0") in {"1", "true", "True", "yes", "YES"}


##**5. Utilidades puras (funciones core)**

- Sin side-effects (no escribir archivos dentro)
- Inputs/outputs claros
- Tipado + docstrings

## **6. Summary / Report (estándar con envelope)**

Funciones recomendadas:
- `build_stage_XX_summary(...) -> dict`
- `save_json(summary, REPORT_SUMMARY)`
- `print_stage_XX_summary_console(summary)`

Envelope común (siempre presente):

In [ ]:
{
  "stage": "stage_01_intraday_data_preparation",
  "created_at_utc": "...",
  "version": "1.0",
  "paths": { "inputs": {...}, "outputs": {...}, "reports": {...} },
  "params": { ... },
  "metrics": { ... },
  "details": { ... }
}

Reglas:
- `params` y `metrics` existen siempre (aunque estén vacíos).
- `details` es libre y específico del stage.
- El JSON de `reports/` es la fuente de verdad del stage.

## **7. Tracking MLflow (formal y desacoplado)**

- MLflow es opcional (`--enable-mlflow` o `ENABLE_MLFLOW`)
- Import perezoso dentro de la función
- MLflow consume `summary["params"]` y `summary["metrics"]`
- Siempre loguear el JSON como artifact

Firma recomendada:

- `mlflow_tracking(summary, *, enable=..., run_name=..., tags=..., artifacts=[REPORT_SUMMARY, ...])`



### **7.1. Convención params / metrics**


**params**

- Config de control y reproducible (horarios, tz, umbrales, horizon, etc.)
- Tipos simples (str/int/float/bool)

**metrics**

- Solo números finales (int/float)
- Nombres estables por stage (no tienen que coincidir entre stages)

Ejemplo (stage_01):



In [ ]:
"params": {
  "market": "NASDAQ",
  "trading_start": "06:30:00",
  "trading_end": "16:00:00",
  "tz_from": "UTC",
  "tz_to": "America/New_York",
  "gap_minutes": 1
},
"metrics": {
  "total_days_raw": 1200,
  "trading_days_output": 1187,
  "discarded_days": 13,
  "discarded_days_pct": 1.0833,
  "records_per_day_median": 451,
  "total_nans": 0
},
"details": {
  "records_per_day": { "min": 450, "median": 451, "max": 451 },
  "trading_time_range_effective": { "start": "...", "end": "..." },
  "days_with_gaps": 0
}

## **8. `parse_args()` (opcional, pero estándar)**

- `argparse` solo como override
- Defaults deben venir de constantes DVC-friendly

Ejemplo:

In [ ]:
parser.add_argument("--raw-path", default=str(IN_RAW_PARQUET))

## **9. `main()` (orquestación)**

- Logs por pasos `[1]`, `[2]`, etc.
- Sin lógica pesada inline: solo orquesta funciones

Orden típico:

1. Cargar + validar
2. Procesar
3. Checks
4. Construir + guardar parquet
5. Construir + guardar summary
6. Print summary
7. MLflow tracking

## **10. Boilerplate**

In [ ]:
if __name__ == "__main__":
    main()

# ----------Esqueleto de script----------------

In [ ]:
"""
stage_XX_template.py

Contrato (Stage Template)
------------------------
Propósito:
- Template profesional y consistente para cualquier stage_XX_* del pipeline.

Inputs (deps DVC):
- IN_* (definidos en "Configuración de rutas")

Outputs (outs DVC):
- OUT_* (definidos en "Configuración de rutas")

Reports (reports/metrics):
- REPORT_* (JSON/HTML livianos para auditoría)

Params:
- PARAM_* (valores reproducibles; env-first; argparse solo como override)

Notas:
- El "summary JSON" en reports/ es la fuente de verdad del stage.
- MLflow es opcional y se alimenta de summary["params"] y summary["metrics"].
"""

from __future__ import annotations

# ---------------------------------------------------------------------
# Imports (stdlib)
# ---------------------------------------------------------------------
import argparse
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Optional

import logging

# ---------------------------------------------------------------------
# Imports (third-party)
# ---------------------------------------------------------------------
# import pandas as pd
# import numpy as np

# ---------------------------------------------------------------------
# Imports (local)
# ---------------------------------------------------------------------
# from neural_profit.utils import ...

# ---------------------------------------------------------------------
# Logging (uniforme)
# ---------------------------------------------------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_XX")

# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly) - SIEMPRE PRESENTE
# ---------------------------------------------------------------------
# Deps (inputs)
IN_INPUT = Path(os.environ.get("IN_INPUT", "data/raw/input.parquet"))

# Outs (outputs)
OUT_OUTPUT = Path(os.environ.get("OUT_OUTPUT", "data/processed/output.parquet"))

# Reports (auditoría)
REPORT_SUMMARY = Path(os.environ.get("REPORT_SUMMARY", "reports/stage_XX_summary.json"))

# ---------------------------------------------------------------------
# Configuración funcional (params reproducibles) - SIEMPRE PRESENTE
# ---------------------------------------------------------------------
ENABLE_MLFLOW = os.environ.get("ENABLE_MLFLOW", "0") in {"1", "true", "True", "yes", "YES"}

# PARAM_... (ejemplos)
PARAM_FOO = os.environ.get("PARAM_FOO", "bar")
PARAM_INT = int(os.environ.get("PARAM_INT", "1"))


# ---------------------------------------------------------------------
# Utilidades generales (reusables)
# ---------------------------------------------------------------------
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def _utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def save_json(payload: Dict[str, Any], output_path: Path) -> None:
    """Guarda un dict como JSON UTF-8 (indentado)."""
    _ensure_parent_dir(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def build_summary_envelope(
    *,
    stage: str,
    version: str,
    inputs: Dict[str, str],
    outputs: Dict[str, str],
    reports: Dict[str, str],
    params: Dict[str, Any],
    metrics: Dict[str, float],
    details: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Construye el summary estándar (envelope) para reports/.
    - params: config reproducible (tipos simples)
    - metrics: solo numéricos (int/float) serializables
    - details: libre (pero JSON-serializable)
    """
    return {
        "stage": stage,
        "created_at_utc": _utc_now_iso(),
        "version": version,
        "paths": {
            "inputs": inputs,
            "outputs": outputs,
            "reports": reports,
        },
        "params": params or {},
        "metrics": metrics or {},
        "details": details or {},
    }


def print_summary_console(summary: Dict[str, Any]) -> None:
    """Imprime un resumen compacto y legible en terminal."""
    stage = summary.get("stage", "N/A")
    created = summary.get("created_at_utc", "N/A")
    metrics = summary.get("metrics", {}) or {}
    params = summary.get("params", {}) or {}

    print("\n" + "=" * 70)
    print(f"STAGE: {stage}")
    print(f"CREATED_AT_UTC: {created}")
    print("-" * 70)

    print("[PARAMS]")
    for k, v in params.items():
        print(f"  - {k}: {v}")

    print("[METRICS]")
    for k, v in metrics.items():
        print(f"  - {k}: {v}")

    print("=" * 70 + "\n")


def mlflow_log_from_summary(
    *,
    enable: bool,
    stage: str,
    summary: Dict[str, Any],
    summary_path: Path,
    run_name: Optional[str] = None,
    tags: Optional[Dict[str, str]] = None,
) -> None:
    """
    Loguea en MLflow:
      - params: summary["params"]
      - metrics: summary["metrics"]
      - artifact: summary JSON
    Si enable=False, no hace nada.
    """
    if not enable:
        return

    try:
        import mlflow
    except ImportError as exc:  # pragma: no cover
        raise ImportError("MLflow no está instalado. Instale con: pip install mlflow") from exc

    if run_name is None:
        run_name = stage

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("stage", stage)
        if tags:
            for k, v in tags.items():
                mlflow.set_tag(k, v)

        # Params
        for k, v in (summary.get("params", {}) or {}).items():
            mlflow.log_param(k, v)

        # Metrics
        for k, v in (summary.get("metrics", {}) or {}).items():
            if isinstance(v, (int, float)) and v == v:  # evita NaN
                mlflow.log_metric(k, float(v))

        # Artifact
        mlflow.log_artifact(str(summary_path))


# ---------------------------------------------------------------------
# Core stage functions (puras) - REEMPLAZAR por la lógica real
# ---------------------------------------------------------------------
def run_stage_logic() -> Dict[str, Any]:
    """
    Ejecuta la lógica principal del stage.

    Recomendación:
    - Esta función NO debería escribir archivos directamente.
    - Debe devolver lo necesario para:
        * escribir OUT_*
        * construir summary (metrics/details)
    """
    # TODO: reemplazar por lógica real
    result: Dict[str, Any] = {
        "dummy_metric": 1.0,
        "dummy_detail": {"note": "replace with real stage logic"},
    }
    return result


# ---------------------------------------------------------------------
# CLI (argparse) - opcional como override
# ---------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Stage template (override via CLI/env).")

    parser.add_argument("--in-input", type=str, default=str(IN_INPUT))
    parser.add_argument("--out-output", type=str, default=str(OUT_OUTPUT))
    parser.add_argument("--report-summary", type=str, default=str(REPORT_SUMMARY))

    parser.add_argument(
        "--enable-mlflow",
        action="store_true",
        default=ENABLE_MLFLOW,
        help="Si se activa, registra params/métricas en MLflow",
    )

    # Ejemplo params override
    parser.add_argument("--param-foo", type=str, default=PARAM_FOO)
    parser.add_argument("--param-int", type=int, default=PARAM_INT)

    return parser.parse_args()


# ---------------------------------------------------------------------
# Main (orquestación)
# ---------------------------------------------------------------------
def main() -> None:
    log.info("[0] Parseando argumentos (CLI/env)")
    args = parse_args()

    in_input = Path(args.in_input)
    out_output = Path(args.out_output)
    report_summary = Path(args.report_summary)

    # Params efectivos (los que realmente se usaron)
    params = {
        "param_foo": args.param_foo,
        "param_int": int(args.param_int),
    }

    log.info("[1] Ejecutando lógica principal del stage")
    result = run_stage_logic()

    # -----------------------------------------------------------------
    # Escritura de OUT_* (ejemplo: placeholder)
    # -----------------------------------------------------------------
    # TODO: reemplazar por escritura real (parquet/json/model/etc.)
    _ensure_parent_dir(out_output)
    # out_output.write_text("placeholder", encoding="utf-8")

    # -----------------------------------------------------------------
    # Construcción summary (envelope)
    # -----------------------------------------------------------------
    metrics = {
        # TODO: reemplazar por métricas reales
        "dummy_metric": float(result.get("dummy_metric", 0.0)),
    }
    details = {
        # TODO: reemplazar por detalles reales (JSON-serializable)
        "dummy_detail": result.get("dummy_detail", {}),
    }

    summary = build_summary_envelope(
        stage="stage_XX_name_here",
        version="1.0",
        inputs={"in_input": str(in_input.as_posix())},
        outputs={"out_output": str(out_output.as_posix())},
        reports={"summary": str(report_summary.as_posix())},
        params=params,
        metrics=metrics,
        details=details,
    )

    log.info("[2] Guardando summary JSON: %s", report_summary)
    save_json(summary, report_summary)

    # (opcional) imprimir en consola
    print_summary_console(summary)

    log.info("[3] MLflow tracking (enable=%s)", args.enable_mlflow)
    mlflow_log_from_summary(
        enable=bool(args.enable_mlflow),
        stage=summary["stage"],
        summary=summary,
        summary_path=report_summary,
        tags={"pipeline": "neural_profit"},
    )

    log.info("[OK] Stage completado")
    log.info("Output: %s", out_output)
    log.info("Summary: %s", report_summary)


# ---------------------------------------------------------------------
# Boilerplate
# ---------------------------------------------------------------------
if __name__ == "__main__":
    main()



# anterior


### **0) Docstring del módulo (contrato del stage)**

- Nombre del stage
- Propósito (1–2 líneas)
- Inputs / Outputs / Reports / Params
- Notas (supuestos y no-objetivos)

### **1) Imports**
- stdlib
- third-party
- local imports
- imports opcionales dentro de funciones (MLflow, etc.)

### **2) Logging (uniforme)**

- LOG_LEVEL por env
- log = logging.getLogger("stage_XX")

### **3) Configuración DVC-friendly (SIEMPRE presente)**

Siempre definir:

- IN_* (deps)
- OUT_* (outs)
- REPORT_* (reports)
- PARAMS (defaults reproducibles)

Ejemplo:

```python
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
IN_RAW_PARQUET  = Path(os.environ.get("IN_RAW_PARQUET", "data/raw/mnq_raw.parquet"))
OUT_PARQUET     = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday.parquet"))
REPORT_SUMMARY  = Path(os.environ.get("REPORT_SUMMARY", "reports/stage_01_dataset_prep_summary.json"))
```

Esto evita que cada stage “invente” paths via argparse y quede inconsistente.


### **4) Configuración funcional (parámetros del stage)**

Bloque “Params” con defaults:

```python
MARKET = os.environ.get("MARKET", "NASDAQ")
TZ_FROM = os.environ.get("TZ_FROM", "UTC")
TZ_TO = os.environ.get("TZ_TO", "America/New_York")
TRADING_START = os.environ.get("TRADING_START", "06:30:00")
TRADING_END = os.environ.get("TRADING_END", "16:00:00")
GAP_MINUTES = int(os.environ.get("GAP_MINUTES", "1"))
```

### **5) Utilidades puras (funciones “core”)**

- Sin side-effects (no escribir archivos dentro)
- Inputs/outputs claros
- Tipado + docstrings

### **6) Summary/Report**

```python
- build_stage_XX_summary(...) -> dict
- save_json(summary, REPORT_SUMMARY)
- print_stage_XX_summary_console(summary)
```
Envelope común (siempre presente):
```json
{
  "stage": "stage_01_intraday_data_preparation",
  "created_at_utc": "...",
  "version": "1.0",
  "paths": { "inputs": {...}, "outputs": {...}, "reports": {...} },
  "params": { ... },          // config del stage (normalizado)
  "metrics": { ... },         // métricas numéricas (normalizado)
  "details": { ... }          // libre, específico del stage
}
```

Claves:

- `params` y `metrics`: estructura estable (aunque cambien las claves internas)
- `details`: cada stage pone lo que necesite (tablas, listas, breakdowns por día/archivo, etc.)




### **7) Tracking MLflow (params/metrics + artifacts)**

- `--enable-mlflow` o `ENABLE_MLFLOW`
- import perezoso dentro de la función

- `mlflow_tracking(summary, *, enable=..., run_name=..., tags=..., artifacts=[REPORT_SUMMARY, ...])

Regla clave:

- El report JSON siempre se genera.
- MLflow es opcional, y si está activo:
    - loguea params (config)
    - loguea metrics (números)
    - loguea artifacts (el JSON del report + opcionalmente muestras/plots)


#### Convención para params y metrics (MLflow-friendly)

- params (configuración)
- Solo cosas “de control” y reproducibles: horarios, tz, umbrales, horizon, etc.
- Tipos simples (str/int/float/bool)

- metrics (números comparables)
- Solo valores numéricos finales (int/float)
- Nombres estables por stage (no hace falta que coincidan entre stages)

Ejemplo stage_01:

```json
"params": {
  "market": "NASDAQ",
  "trading_start": "06:30:00",
  "trading_end": "16:00:00",
  "tz_from": "UTC",
  "tz_to": "America/New_York",
  "gap_minutes": 1
},
"metrics": {
  "total_days_raw": 1200,
  "trading_days_output": 1187,
  "discarded_days": 13,
  "discarded_days_pct": 1.0833,
  "records_per_day_median": 451,
  "total_nans": 0
},
"details": {
  "records_per_day": { "min": 450, "median": 451, "max": 451 },
  "trading_time_range_effective": { "start": "...", "end": "..." },
  "days_with_gaps": 0
}
```


#### **8) parse_args() (opcional, pero estandarizado)**

Mi recomendación:
- Mantener argparse solo como override
- Defaults deben venir de los constantes DVC-friendly definidos arriba

Ejemplo:
```python
parser.add_argument("--in-raw", default=str(IN_RAW_PARQUET))
```

**9) main() (orquestación)**

- logging por pasos `[1]`, `[2]`, etc.
- no lógica pesada inline: usar funciones

10) Boilerplate

- `if __name__ == "__main__": main()`

In [ ]:
def mlflow_log_from_summary(*, enable: bool, stage: str, summary: dict, summary_path: Path) -> None:
    if not enable:
        return
    try:
        import mlflow
    except ImportError as exc:
        raise ImportError("MLflow no está instalado. Instale con: pip install mlflow") from exc

    with mlflow.start_run(run_name=stage):
        mlflow.set_tag("stage", stage)

        for k, v in (summary.get("params", {}) or {}).items():
            mlflow.log_param(k, v)

        for k, v in (summary.get("metrics", {}) or {}).items():
            if isinstance(v, (int, float)) and v == v:  # evita NaN
                mlflow.log_metric(k, float(v))

        mlflow.log_artifact(str(summary_path))